# Simulation Walkthrough — `tests.py` Scenarios

Each section runs one scenario from the legacy `tests.py` file in order:

1. **Flat Sheet Solver (baseline)** — basic vertex-model on the 2-D planar tissue, short run  
2. **Flat Sheet Solver** — same vertex model on a 2-D planar tissue  
3. **Regulation (Cell Division)** — periodic division events  
4. **Stochastic Line Tension** — Ornstein-Uhlenbeck noise on edge tensions  
5. **Cell Jamming** — stochastic tension + a jamming transition at t = 100  
6. **Parameter Gradient** — stochastic tension + a linear perimeter gradient  
7. **Anisotropic Tension** — axis-aligned tension anisotropy  
8. **Gillespie Crypt** — full Gillespie biochemistry driving growth/division/apoptosis on a crypt cylinder  

> **Working directory note:** run this notebook from the `Notebooks/` folder  
> (or execute the `%cd` cell below) so the `.hf5` tissue files are found.

## 0. Setup

In [ ]:
import os, pathlib
# Ensure the kernel CWD is Notebooks/ so .hf5 paths resolve
nb_dir = pathlib.Path("__file__" if "__file__" in dir() else ".").resolve().parent
if nb_dir.name == "Notebooks":
    os.chdir(nb_dir)
elif (nb_dir / "Notebooks").exists():
    os.chdir(nb_dir / "Notebooks")
print("Working directory:", os.getcwd())

In [ ]:
import numpy as np
import time
import pandas as pd
from pprint import pprint
from IPython.display import Image, display

from matplotlib import pyplot as plt
from bigraph_schema import allocate_core
from bigraph_viz import plot_bigraph
from process_bigraph import Process, Composite
from process_bigraph.emitter import emitter_from_wires, gather_emitter_results

from vivarium_tyssue.data_types import register_types
from vivarium_tyssue.processes import register_processes, EulerSolver
from vivarium_tyssue.processes.regulations import (
    TestRegulations, StochasticLineTension, CellJamming,
    ParameterGradient, AnisotropicTension,
)
from vivarium_tyssue.processes.gillespie import Gillespie
from vivarium_tyssue.models.crypt_gillespie.crypt_params import *
from vivarium_tyssue.models.crypt_gillespie.jump_rates import *
from vivarium_tyssue.draw import (
    CELL_TYPE_COLORS, cell_type_kwds, crypt_cell_type_kwds,
)

from tyssue import config
from tyssue.draw import create_gif, create_gif_3d

In [ ]:
# Initialise the core and register all types + processes
core = allocate_core()
core = register_types(core)
core = register_processes(core)
print("Core ready.")

### Helper functions (ported verbatim from `tests.py`)

In [ ]:
def get_test_config():
    return {
        "name": "Test Cylinder",
        "eptm": "test_cylinder.hf5",
        "tissue_type": "Sheet",
        "parameters": {
            "face_df": {
                "area_elasticity": 1.0,
                "prefered_area": 1.0,
                "perimeter_elasticity": 0.5,
                "prefered_perimeter": 3.5,
            },
            "edge_df": {
                "line_tension": 0.0,
                "is_active": 1.0,
            },
            "vert_df": {
                "viscosity": 0.1,
                "vessel_elasticity": 1.0,
                "prefered_radius": 2.5,
                "is_alive": 1.0,
            },
        },
        "geom": "VesselGeometry",
        "effectors": ["LineTension", "FaceAreaElasticity", "PerimeterElasticity", "VesselSurfaceElasticity"],
        "ref_effector": "FaceAreaElasticity",
        "factory": "model_factory",
        "settings": {"threshold_length": 0.03},
        "auto_reconnect": True,
        "bounds": None,
        "output_columns": {},
        "maps": {},
    }


def get_test_config_flat():
    return {
        "name": "Test Square",
        "eptm": "test_square.hf5",
        "tissue_type": "Sheet",
        "parameters": {
            "face_df": {
                "area_elasticity": 1,
                "prefered_area": 1,
                "perimeter_elasticity": 0.1,
                "prefered_perimeter": 3.6,
                "migration_strength": [0.1 if i == 33 else 0.0 for i in range(206)],
                "is_alive": 1,
                "mx": 1,
                "mz": 0,
                "my": 1,
            },
            "edge_df": {
                "line_tension": 0,
                "is_active": 1,
            },
            "vert_df": {
                "viscosity": 1,
                "is_alive": 1,
            },
        },
        "geom": "SheetGeometry",
        "effectors": [
            "LineTension",
            "FaceAreaElasticity",
            "PerimeterElasticity",
            # "ActiveMigration"  # commented out in original
        ],
        "ref_effector": "FaceAreaElasticity",
        "factory": "model_factory_bound",
        "settings": {"threshold_length": 0.03},
        "auto_reconnect": True,
        "bounds": None,
        "output_columns": {},
    }

In [ ]:
def get_test_spec(interval=0.1, config=None):
    return {
        "Tyssue": {
            "_type": "process",
            "address": "local:EulerSolver",
            "config": config,
            "inputs": {
                "behaviors": ["Behaviors"],
                "global_time": ["global_time"],
            },
            "outputs": {
                "datasets": ["Datasets"],
                "network_changed": ["Network Changed"],
                "behaviors_update": ["Behaviors"],
            },
            "interval": interval,
        },
        "Network Changed": False,
        "Behaviors": {},
    }


def get_test_regulation_spec(interval=0.1, config=None, double=False):
    spec = get_test_spec(interval=interval, config=config)
    spec["Regulation"] = {
        "_type": "process",
        "address": "local:TestRegulations",
        "config": {
            "period": 5,
            "geom": "SheetGeometry",
            "crit_area": 2,
            "growth_rate": 0.2,
            "double": double,
        },
        "inputs": {
            "global_time": ["global_time"],
            "datasets": ["Datasets"],
        },
        "outputs": {
            "behaviors": ["Behaviors"],
        },
        "interval": interval,
    }
    return spec


def get_test_stochastic_spec(interval=0.1, config=None, tau=1.0, sigma=1.0):
    spec = get_test_spec(interval=interval, config=config() if callable(config) else config)
    spec["Stochastic"] = {
        "_type": "process",
        "address": "local:StochasticLineTension",
        "config": {"tau": tau, "sigma": sigma},
        "inputs":  {"datasets": ["Datasets"]},
        "outputs": {"behaviors": ["Behaviors"]},
        "interval": interval,
    }
    return spec


def get_test_jamming_spec(interval=0.1, config=None, tau=1.0, sigma=1.0):
    spec = get_test_stochastic_spec(interval=interval,
                                    config=config() if callable(config) else config,
                                    tau=tau, sigma=sigma)
    spec["Jamming"] = {
        "_type": "process",
        "address": "local:CellJamming",
        "config": {
            "trigger_time": 100,
            "rate": -0.05,
            "limits": [3.0, 4.2],
        },
        "inputs":  {"global_time": ["global_time"], "datasets": ["Datasets"]},
        "outputs": {"behaviors": ["Behaviors"]},
        "interval": interval,
    }
    return spec


def get_test_gradient_spec(interval=0.1, config=None, tau=1.0, sigma=1.0):
    spec = get_test_stochastic_spec(interval=interval,
                                    config=config() if callable(config) else config,
                                    tau=tau, sigma=sigma)
    spec["Gradient"] = {
        "_type": "step",
        "address": "local:ParameterGradient",
        "config": {
            "gradient_type": "linear",
            "axis": "x",
            "args": {"m": -0.1, "c": 4.6},
            "model_parameters": {"prefered_perimeter": "face"},
        },
        "inputs":  {"datasets": ["Datasets"]},
        "outputs": {"behaviors": ["Behaviors"]},
    }
    return spec


def get_test_anisotropic_spec(interval=0.1, config=None):
    spec = get_test_spec(interval=interval, config=config() if callable(config) else config)
    spec["Anisotropic"] = {
        "_type": "step",
        "address": "local:AnisotropicTension",
        "config": {
            "axes": ["x", "y"],
            "tension_values": [0, 0.2],
        },
        "inputs":  {"datasets": ["Datasets"]},
        "outputs": {"behaviors": ["Behaviors"]},
    }
    return spec


def get_test_gillespie_config(
    interval=0.1,
    growth_rate=0.02,
    shrink_rate=0.02,
    division_crit=1.2,
    apoptosis_crit=0.1,
):
    return {
        "cell_types": cell_types,
        "rates_max": rates_max,
        "michaelis_constants": K,
        "transition_lengths": k,
        "geom": "VesselGeometry",
        "global_interval": interval,
        "growth_rate": growth_rate,
        "shrink_rate": shrink_rate,
        "division_crit": division_crit,
        "apoptosis_crit": apoptosis_crit,
        "regulations": regulations,
        "regulation_loc": regulation_loc,
    }


def base_gillespie_spec(interval=0.1):
    return {
        "Gillespie": {
            "_type": "process",
            "address": "local:Gillespie",
            "config": get_test_gillespie_config(),
            "inputs": {
                "datasets": ["Datasets"],
                "behaviors": ["Behaviors"],
                "global_time": ["global_time"],
            },
            "outputs": {
                "behaviors": ["Behaviors"],
                "gillespie_trigger": ["Gillespie Trigger"],
            },
            "interval": interval,
        }
    }


def get_test_gillespie_spec(interval=0.01, config=None, tau=1.0, sigma=1.0):
    spec = get_test_spec(interval=interval, config=config() if callable(config) else config)
    spec["Tyssue"]["config"]["eptm"] = "crypt_cylinder.hf5"
    spec["Tyssue"]["config"]["settings"].update({"radius": 2.5, "axis": "z"})
    spec["Tyssue"]["config"]["geom"] = "VesselGeometry"
    spec["Tyssue"]["config"]["factory"] = "model_factory_vessel"
    spec["Tyssue"]["config"]["effectors"] = [
        "FaceAreaElasticity", "PerimeterElasticity",
        "LineTension", "VesselSurfaceElasticity",
    ]
    spec["Tyssue"]["config"]["parameters"]["vert_df"]["viscosity"] = 0.05
    spec["Tyssue"]["config"]["parameters"]["vert_df"]["vessel_elasticity"] = 0.1
    spec.update(base_gillespie_spec(interval=interval))
    return spec

In [ ]:
# Shared emitter used by most run_test_* helpers
test_emitter = emitter_from_wires({
    "global_time": ["global_time"],
    "face_df":     ["Datasets", "face_df"],
    "edge_df":     ["Datasets", "edge_df"],
    "vert_df":     ["Datasets", "vert_df"],
    "behaviors":   ["Behaviors"],
})

### Common draw-spec helpers

In [ ]:
def base_draw_specs(face_color="blue", edge_color="black", face_alpha=0.5):
    ds = config.draw.sheet_spec()
    ds["face"]["visible"] = True
    ds["face"]["alpha"]   = face_alpha
    ds["face"]["color"]   = face_color
    ds["edge"]["color"]   = edge_color
    ds["edge"]["width"]   = 0.5
    ds["edge"]["alpha"]   = 0.8
    return ds


def flat_draw_specs(highlight_face=96, n_faces=206, alpha=0.5):
    """Autumn-cmap specs highlighting a single cell (used for flat-sheet sims)."""
    ds = config.draw.sheet_spec()
    cmap = plt.get_cmap("autumn")
    color_map = cmap([1.0 if i == highlight_face else 0.0 for i in range(n_faces)])
    ds["face"]["visible"] = True
    ds["face"]["alpha"]   = alpha
    ds["face"]["color"]   = color_map
    ds["edge"]["color"]   = "black"
    return ds

---
## 1. Flat Sheet Solver (baseline)

Basic vertex-model simulation on the **2-D planar (hexagonal) tissue** (`test_square.hf5`).  
Energy function uses area elasticity, perimeter elasticity and line tension,  
with the bound `model_factory_bound` factory and `SheetGeometry`. A short  
20-step run for a quick sanity check before the longer runs below.

In [ ]:
config1 = get_test_config_flat()
spec1 = get_test_spec(interval=0.1, config=config1)
spec1["emitter"] = emitter_from_wires({
    "global_time": ["global_time"],
    "face_df":     ["Datasets", "face_df"],
    "edge_df":     ["Datasets", "edge_df"],
    "vert_df":     ["Datasets", "vert_df"],
})
plot_bigraph(spec1, core=core, out_dir="bigraphs", filename="01_flat_baseline")

In [ ]:
start = time.time()
sim1 = Composite({"state": spec1}, core=core)
sim1.run(20)
results1 = gather_emitter_results(sim1)[("emitter",)]
print(f"Done in {time.time() - start:.1f}s  |  {len(results1)} time-steps recorded")

In [ ]:
history1 = sim1.state["Tyssue"]["instance"].history
history1.update_datasets()

cmap = plt.get_cmap("autumn")
color_map1 = cmap([0.0 for i in range(206)])
ds1 = base_draw_specs(face_color=color_map1, face_alpha=0.5)
create_gif(history1, "test.gif", coords=["x", "y"], **ds1)
display(Image(filename="test.gif"))

---
## 2. Flat Sheet Solver

Same vertex model on a **2-D planar (hexagonal) tissue** (`test_square.hf5`).  
No vessel term; the factory is `model_factory_bound`.  
Cell 33 gets a small `migration_strength` (originally intended for an  
`ActiveMigration` effector — commented out in the original).

*(This block was commented out in `tests.py`.)*

In [ ]:
config2 = get_test_config_flat()
spec2 = get_test_spec(interval=0.1, config=config2)
spec2["emitter"] = test_emitter
plot_bigraph(spec2, core=core, out_dir="bigraphs", filename="02_flat_sheet")

In [ ]:
start = time.time()
sim2 = Composite({"state": spec2}, core=core)
sim2.run(40)
results2 = gather_emitter_results(sim2)[("emitter",)]
print(f"Done in {time.time() - start:.1f}s  |  {len(results2)} time-steps")

In [ ]:
history2 = sim2.state["Tyssue"]["instance"].history
history2.update_datasets()

# Colour cell 33 (highlight_face) with autumn cmap; all others at 0
ds2 = config.draw.sheet_spec()
cmap = plt.get_cmap("autumn")
color_map2 = cmap([0.0 for i in range(206)])
ds2["face"]["visible"] = True
ds2["face"]["alpha"]   = 0.5
ds2["face"]["color"]   = color_map2
ds2["edge"]["color"]   = "black"

create_gif(history2, "test_flat.gif", coords=["x", "y"], num_frames=200, **ds2)
display(Image(filename="test_flat.gif"))

---
## 3. Regulation — Cell Division

A `TestRegulations` process fires every `period=5` time units and triggers  
cell **growth** (area target increases at `growth_rate=0.2`) followed by  
**division** once a cell exceeds `crit_area=2`.

*(This block was commented out in `tests.py`.)*

In [ ]:
spec3 = get_test_regulation_spec(interval=0.1, config=get_test_config_flat(), double=False)
plot_bigraph(spec3, core=core, out_dir="bigraphs", filename="base_bigraph")

In [ ]:
spec3["emitter"] = test_emitter
plot_bigraph(spec3, core=core, out_dir="bigraphs", filename="03_regulation_division")

In [ ]:
start = time.time()
sim3 = Composite({"state": spec3}, core=core)
sim3.run(40)
results3 = gather_emitter_results(sim3)[("emitter",)]
print(f"Done in {time.time() - start:.1f}s  |  {len(results3)} time-steps")

In [ ]:
history3 = sim3.state["Tyssue"]["instance"].history
history3.update_datasets()
cmap = plt.get_cmap("autumn")
color_map3 = cmap([0.0 for i in range(206)])
ds3 = base_draw_specs(face_color=color_map3, face_alpha=0.5)
create_gif(history3, "test_division.gif", coords=["x", "y"], **ds3)
display(Image(filename="test_division.gif"))

---
## 4. Stochastic Line Tension

An Ornstein-Uhlenbeck process (`StochasticLineTension`) adds correlated  
noise to edge line tensions (`τ=0.2`, `σ=0.1`).  
Run on the flat sheet with a single highlighted migrating cell.

*(This block was commented out in `tests.py`.)*

In [ ]:
spec4 = get_test_stochastic_spec(
    interval=0.1, config=get_test_config_flat(), tau=0.2, sigma=0.1
)
spec4["Tyssue"]["config"]["effectors"].append("ActiveMigration")
spec4["emitter"] = test_emitter
plot_bigraph(spec4, core=core, out_dir="bigraphs", filename="04_stochastic_tension")

In [ ]:
start = time.time()
sim4 = Composite({"state": spec4}, core=core)
sim4.run(100)
results4 = gather_emitter_results(sim4)[("emitter",)]
print(f"Done in {time.time() - start:.1f}s  |  {len(results4)} time-steps")

In [ ]:
history4 = sim4.state["Tyssue"]["instance"].history
history4.update_datasets()

cmap = plt.get_cmap("autumn")
color_map4 = cmap([1.0 if i == 33 else 0.0 for i in range(206)])
ds4 = config.draw.sheet_spec()
ds4["face"]["visible"] = True
ds4["face"]["alpha"]   = 0.5
ds4["face"]["color"]   = color_map4
ds4["edge"]["color"]   = "black"

create_gif(history4, "test_stochastic_migration.gif", coords=["x", "y"],
           num_frames=200, **ds4)
display(Image(filename="test_stochastic_migration.gif"))

---
## 5. Cell Jamming

Same stochastic line-tension setup, plus a `CellJamming` process that  
kicks in at `trigger_time=100` and gradually decreases preferred perimeter  
(`rate=-0.05`) until a jamming limit `[3.0, 4.2]` is reached.

*(This block was commented out in `tests.py`.)*

In [ ]:
spec5 = get_test_jamming_spec(
    interval=0.1, config=get_test_config_flat(), tau=0.2, sigma=0.1
)
spec5["Tyssue"]["config"]["effectors"].append("ActiveMigration")
spec5["emitter"] = test_emitter
plot_bigraph(spec5, core=core, out_dir="bigraphs", filename="05_cell_jamming")

In [ ]:
start = time.time()
sim5 = Composite({"state": spec5}, core=core)
sim5.run(200)
results5 = gather_emitter_results(sim5)[("emitter",)]
print(f"Done in {time.time() - start:.1f}s  |  {len(results5)} time-steps")

In [ ]:
history5 = sim5.state["Tyssue"]["instance"].history
history5.update_datasets()

cmap = plt.get_cmap("autumn")
color_map5 = cmap([1.0 if i == 33 else 0.0 for i in range(206)])
ds5 = config.draw.sheet_spec()
ds5["face"]["visible"] = True
ds5["face"]["alpha"]   = 0.5
ds5["face"]["color"]   = color_map5
ds5["edge"]["color"]   = "black"

create_gif(history5, "test_jamming_flat.gif", coords=["x", "y"],
           num_frames=200, **ds5)
display(Image(filename="test_jamming_flat.gif"))

---
## 6. Parameter Gradient

Stochastic line tension + a `ParameterGradient` step that applies a  
**linear gradient** along `x` to each cell's `prefered_perimeter`:  
`p0 = -0.1·x + 4.6`. Cell 96 also gets a migration push (`migration_strength=0.1`).

*(This block was commented out in `tests.py`.)*

In [ ]:
spec6 = get_test_gradient_spec(
    interval=0.05, config=get_test_config_flat, tau=0.2, sigma=0.1
)
spec6["Tyssue"]["config"]["parameters"]["face_df"]["migration_strength"] = (
    [0.1 if i == 96 else 0.0 for i in range(206)]
)
spec6["Tyssue"]["config"]["parameters"]["face_df"]["my"] = 0
spec6["Tyssue"]["config"]["effectors"].append("ActiveMigration")
spec6["emitter"] = test_emitter
plot_bigraph(spec6, core=core, out_dir="bigraphs", filename="06_parameter_gradient")

In [ ]:
start = time.time()
sim6 = Composite({"state": spec6}, core=core)
sim6.run(200)
results6 = gather_emitter_results(sim6)[("emitter",)]
print(f"Done in {time.time() - start:.1f}s  |  {len(results6)} time-steps")

In [ ]:
history6 = sim6.state["Tyssue"]["instance"].history
history6.update_datasets()

cmap = plt.get_cmap("autumn")
color_map6 = cmap([1.0 if i == 96 else 0.0 for i in range(206)])
ds6 = config.draw.sheet_spec()
ds6["face"]["visible"] = True
ds6["face"]["alpha"]   = 0.5
ds6["face"]["color"]   = color_map6
ds6["edge"]["color"]   = "black"

create_gif(history6, "test_gradient.gif", coords=["x", "y"], num_frames=200, **ds6)
display(Image(filename="test_gradient.gif"))

---
## 7. Anisotropic Tension

An `AnisotropicTension` step sets higher line tension on edges aligned with  
`y` (`tension_values=[0, 0.2]`), producing elongated cells along `x`.  
The `factory` is overridden to `model_factory` (removes the bound constraint).

In [ ]:
spec7 = get_test_anisotropic_spec(interval=0.1, config=get_test_config_flat)
spec7["Tyssue"]["config"]["factory"] = "model_factory"
spec7["Tyssue"]["config"]["parameters"]["face_df"]["prefered_perimeter"] = 3.4
spec7["emitter"] = test_emitter
plot_bigraph(spec7, core=core, out_dir="bigraphs", filename="07_anisotropic_tension")

In [ ]:
start = time.time()
sim7 = Composite({"state": spec7}, core=core)
sim7.run(20)
results7 = gather_emitter_results(sim7)[("emitter",)]
print(f"Done in {time.time() - start:.1f}s  |  {len(results7)} time-steps")

In [ ]:
history7 = sim7.state["Tyssue"]["instance"].history
history7.update_datasets()

cmap = plt.get_cmap("autumn")
color_map7 = cmap([0.0 for i in range(206)])
ds7 = config.draw.sheet_spec()
ds7["face"]["visible"] = True
ds7["face"]["alpha"]   = 0.5
ds7["face"]["color"]   = color_map7
ds7["edge"]["color"]   = "black"

create_gif(history7, "test_anisotropic_behavior.gif", coords=["x", "y"],
           num_frames=200, **ds7)
display(Image(filename="test_anisotropic_behavior.gif"))

---
## 8. Gillespie Crypt

Full Gillespie biochemistry (`Gillespie` process) on the 3-D crypt cylinder  
(`crypt_cylinder.hf5`). The Gillespie process tracks each cell's type  
(`sc`, `pc`, `ent`, `gc`) and drives:
- area **growth** / **shrinkage** based on Michaelis-Menten kinetics  
- **division** when area exceeds `division_crit`  
- **apoptosis** when area drops below `apoptosis_crit`  

`tf=72`, `dt=0.005`. The history is saved to `gillespie_history.hf5`.  
The `create_gif_3d` block was commented out in the original — included here  
for completeness.

*(Gillespie run was active in `tests.py`; `create_gif_3d` was commented out.)*

In [ ]:
spec8 = get_test_gillespie_spec(interval=0.005, config=get_test_config)
spec8["emitter"] = emitter_from_wires({
    "global_time": ["global_time"],
    "behaviors":   ["Behaviors"],
})
plot_bigraph(spec8, core=core, out_dir="bigraphs", filename="08_gillespie_crypt")

In [ ]:
start = time.time()
sim8 = Composite({"state": spec8}, core=core)
sim8.run(72)
results8 = gather_emitter_results(sim8)[("emitter",)]
print(f"Done in {time.time() - start:.1f}s  |  {len(results8)} time-steps")

In [ ]:
history8 = sim8.state["Tyssue"]["instance"].history
history8.update_datasets()
history8.to_archive("gillespie_history.hf5")
print("Archive saved to gillespie_history.hf5")

In [ ]:
# 3-D GIF with per-cell-type colouring (was commented out in tests.py)
ds8 = config.draw.sheet_spec()
ds8["face"]["visible"] = True
ds8["face"]["alpha"]   = 1.0
ds8["edge"]["color"]   = "black"

create_gif_3d(
    history8,
    output="test_gillespie.gif",
    num_frames=144,
    coords=["x", "y", "z"],
    dynamic_draw_kwds=[crypt_cell_type_kwds],
    legend=CELL_TYPE_COLORS,
    cull_back_edges=True,
    **ds8,
)
display(Image(filename="test_gillespie.gif"))